In [1]:
import os 

In [2]:
os.chdir("..\.")

In [3]:
import torch 
import torch.nn.functional as F

In [4]:
torch.manual_seed(1337)
with open ("data\gpt_train.txt","r") as file:
    text    = file.read()


# All the unique characters that occur in this text 
chars       = sorted((set(text)))
vocab_size  = len(chars)

## creating mapping from characters to integers 
stoi    = {ch:i for i,ch in enumerate(chars)}
itos    = dict(enumerate(chars))
encode  = lambda word: [stoi[i] for i in word]
decode  = lambda integers: "".join(itos[int(i)] for i in integers)



## Let's encode entire dataset of file. 
data = torch.tensor(encode(text),dtype=torch.long,device="cuda")

## Lets split the data into train and val dataset 
n   = int(0.9 * len(data))
train_data  = data[:n]
val_data    = data[n:]

## FeedForward Layer 

- A feedforward layer in a Transformer block is not just a single layer but typically a small neural network itself, consisting of two linear transformations (fully connected layers) with a non-linear activation function (usually ReLU or GELU) in between.
- Another word: adding comutation into network. 


**In Multi-head self-attention layer they just did the communication between the tokens, but they forgot to think on what they foung from the other tokens.*

### The key characteristics are:

**1. Position-wise Application:** 
    - This is a critical distinction. Unlike the multi-head attention mechanism which considers the entire sequence to compute contextual representations, the FFN is applied independently and identically to each position (e.g., each word's vector representation) in the sequence.
    - If you have a sequence of N tokens, each token's vector representation passes through the same FFN, but the computations for each token are independent of the others. This parallelism is a significant advantage of Transformers.

**2. Expansion and Contraction:**

- The first linear layer typically expands the dimensionality of the input vector.
    - Example:  if the input vector for a token has a dimension of dmodel​ (e.g., 512), the first linear layer might project it to a much larger dimension, say dff​ (e.g., 2048 ==> 4*512 ).
- The non-linear activation function is applied to this expanded representation.
- The second linear layer then contracts this expanded representation back to the original dmodel​ dimensionality.

- Mathematically, for an input x to the FFN (which is a vector representing a single token's output from the attention sublayer):
    - FFN(x)=max(0,xW1​+b1​)W2​+b2​

- where W1​,b1​,W2​,b2​ are learnable weight matrices and bias vectors, and max(0,⋅) is the ReLU activation function (though GELU is often used in modern Transformers).


**3. Residual Connection and Layer Normalization:**

- Like the self-attention sub-layer, the FFN sub-layer in a Transformer is usually followed by a residual connection (adding the input of the sub-layer to its output) and then layer normalization. This helps with training stability and allows for deeper networks.

In [5]:
import torch
from tqdm import tqdm
from torch import nn
from torch.nn import functional as F
from common.data_processing import get_batch     


batch_size  = 4     # B 
block_size  = 8     # T
n_emd       = 32    # C 
device      = "cuda" if torch.cuda.is_available() else "cpu"
eval_interval   = 300
learning_rate   = 1e-3 
max_iters       = 5000 
eval_iter       = 200

vocab_size      = 65 

In [6]:
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.key    = nn.Linear(n_emd,head_size,bias=False)   # key projection
        self.query  = nn.Linear(n_emd,head_size,bias=False)   # query projection
        self.value  = nn.Linear(n_emd,head_size,bias=False)   # value projection 
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
    def forward(self,x):
        _,T,C   = x.shape
        k       = self.key(x)   # (B,T,head_size) ==> 4,8,16
        q       = self.query(x) # (B,T,head_size) ==> 4,8,16   
        ## compute attention score(affinities)
        wei     = q @ k.transpose(-2,-1) * C ** -0.5   # (B,T,c) @ (B,C,T) ==> (B,T,T)
        wei     = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei     = F.softmax(wei,dim=-1)
        # perform the weighted aggregation of the value
        v       = self.value(x) # (B,T,head_size) ==> 4,8,16
        out     = wei @ v       # (B,T,T) @ (B,T,head_size) ==> (B,T,head_size) 
        return out 

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads,head_size):
        super().__init__()
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    def forward(self,x):
        return torch.cat([head(x) for head in self.heads],dim=-1)   # concatenate in the channel dimension 

In [8]:
class FeedForward(nn.Module):
    def __init__(self,n_emd):
        super().__init__()
        self.net    = nn.Sequential(
            nn.Linear(n_emd,n_emd),
            nn.ReLU()
        )
    def forward(self,x):
        return self.net(x)

In [9]:
## make a block: 
class Block(nn.Module):
    def __init__(self,n_emd,n_heads):
        super().__init__()
        head_size                   = n_emd // n_heads 
        self.self_attention_head    = MultiHeadAttention(n_heads,head_size)
        self.ffwd                   = FeedForward(n_emd)
    def forward(self,x):
        x   = self.self_attention_head(x)
        x   = self.ffwd(x)
        return x 

In [10]:
block   = Block(n_emd,n_heads=4)
x       = torch.randn(batch_size,batch_size,n_emd)
block(x).shape

torch.Size([4, 4, 32])

In [ ]:
class BiGramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for next token from a lookup table 
        self.token_embedding_table      = nn.Embedding(vocab_size,n_emd)
        self.position_embedding_table   = nn.Embedding(block_size,n_emd)
        self.block                      = nn.Sequential(
            Block(n_emd,n_heads=4),
            Block(n_emd,n_heads=4),
            Block(n_emd,n_heads=4),
            Block(n_emd,n_heads=4),
        )
        self.lm_head                    = nn.Linear(n_emd,vocab_size)       
        

    def forward(self,idx,target=None):
        B,T         = idx.shape 
        # index and target are both (B,T) tensor of integers
        tok_emb     = self.token_embedding_table(idx)   # its arrange in the shape of ==================================================================>> (B,T,C)
        pos_emb     = self.position_embedding_table(torch.arange(T,device=device)) # This embdding gives the idea of where word is belong in a sentence=>> (T,C) 
        x           = tok_emb + pos_emb                 # combined representation of token and its position.======Broadcasting apply=>> (B,T,C) + (T,C) == (B,T,C) 
        x           = self.block(x)                     # 
        logits      = self.lm_head(x)                   # for getting token_emb to logits we need linear layer,shape ===================================>> (B,T,vocab_size) 

        if target is None:
            loss    = None
        else:    
            B,T,C   = logits.shape
            logits  = logits.view(B*T,C)    # cross entropy input expectation is (minibatch,C)
            target  = target.view(B*T)      
            loss    = nn.functional.cross_entropy(logits,target)
        return logits,loss
    
    def generate(self,idx,max_new_tokens):
        # idx is (B,T) array of indices in the current context. 
        for _ in range(max_new_tokens):
            idx_cond    = idx[:,-block_size:]                   # crop idx to last block_size tokens
            logits,loss = self.forward(idx_cond)                # Get the prediction ==>  (B,T,C)
            logits      = logits[:,-1,:]                        # focus only on last time step  ==>  (B,C)
            probs       = nn.functional.softmax(logits,dim=-1)  # (B,C)
            # sample from the distribution 
            idx_next    = torch.multinomial(probs,num_samples=1)    # (B,1)
            # append sample index to running sequence 
            idx         = torch.cat([idx,idx_next],dim=1)           # (B,T+1)
        return idx


In [ ]:
model = BiGramLanguageModel()
model = model.to("cuda:0")

In [ ]:

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train","val"]:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            X,Y         = get_batch("train")
            logits,loss = model(X,Y)
            losses[k]   = loss.item()   
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:

optimizer   = torch.optim.AdamW(model.parameters(),lr=learning_rate)


for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses  = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f},val loss {losses['val']:.4f}")

    #sample a batch of data
    xb,yb = get_batch("train")
    #evaluate the loss
    logits,loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1,1),dtype=torch.long,device="cuda:0")
print(decode(model.generate(context,max_new_tokens=500)[0]))

step 0: train loss 4.1884,val loss 4.1886
step 300: train loss 3.1229,val loss 3.1434
step 600: train loss 2.8853,val loss 2.8756
step 900: train loss 2.7891,val loss 2.7513
step 1200: train loss 2.6293,val loss 2.6581
step 1500: train loss 2.6091,val loss 2.5940
step 1800: train loss 2.5860,val loss 2.5369
step 2100: train loss 2.5002,val loss 2.5547
step 2400: train loss 2.5098,val loss 2.4996
step 2700: train loss 2.4883,val loss 2.4779
step 3000: train loss 2.4499,val loss 2.4467
step 3300: train loss 2.4680,val loss 2.4667
step 3600: train loss 2.4377,val loss 2.4224
step 3900: train loss 2.4073,val loss 2.4060
step 4200: train loss 2.3737,val loss 2.3949
step 4500: train loss 2.3850,val loss 2.3820
step 4800: train loss 2.3856,val loss 2.3968

And thik bridcowd, har lo, ber

Hisen bobe ule.
SagO-3ISM:

KITanss ar bthat uwqort vet?
Wedthas ate awche my.

HDER:
An otou waowts, tof is he cove whed;
Whoues iree sengcin lat Het drov the and Win now onderjbous lolind peave loull cech b